In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer
from sklearn.metrics import confusion_matrix, accuracy_score
import os
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
stem = PorterStemmer().stem
import string
from scipy.sparse import csr_matrix

In [ ]:
from scipy.sparse import hstack, vstack

In [ ]:
for dirname, _, filenames in os.walk('./kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

### EDA

In [ ]:
df = pd.read_csv('./kaggle/input/comment-category-prediction-challenge/train.csv')

In [ ]:
df = df.replace('none',np.nan)

In [ ]:
df.isnull().sum()

In [ ]:
df['gender'].unique(), df['race'].unique(), df['religion'].unique()

#### Null values Analysis
As comment is the most important feature here, the one row without comment can be dropped
As for race, gender, and religion, 'none' to be replaced with nan

In [ ]:
df.columns

In [ ]:
df.if_1.sort_values().unique()

In [ ]:
df['emoticon_1'].unique()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

hm = pd.concat([
    pd.get_dummies(df['label']).astype(int),
    pd.get_dummies(df['gender']).astype(int),
    pd.get_dummies(df['religion']).astype(int),
    pd.get_dummies(df['race']).astype(int),
    df[['emoticon_1', 'emoticon_2', 'emoticon_3', 'upvote', 'downvote','disability','if_1', 'if_2']]], 
          axis=1).corr() \
    .iloc[[0,1,2,3],4:].transpose()

sns.heatmap(hm, annot=True, cmap='coolwarm',)
plt.show()

#### Features to Ignore
1. Created_date: Although time of comment can be argued to be of some use, it's a long shot
2. post_id
3. emoticons: not very clear what the values represent. Intensity of the feature, or index of some value of the feature.
    (While milestones did consider these to be intensity/value, the correlation is not enough to consider these as valid features)

In [ ]:
df['label'].plot(kind='hist')

### Data Preprocessing

In [ ]:
stop_words = stopwords.words('english')

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = str(text).translate(str.maketrans(string.punctuation,' '*len(string.punctuation)))
    text = str(text).translate(str.maketrans('','', string.digits))    
    text = text.split()
    text = " ".join([stem(i) for i in text]) # if len(i)>2 and len(i)<15]  #  if i not in stop_words
    return text


In [ ]:
# Dropping Unnecessary columns, and rows
df = df.drop(['post_id', 'created_date'], axis=1)
df = df.drop(df[df['comment'].isnull()].index).reset_index(drop=True)

In [ ]:
# Replacing 'none' with nan
# df[['race', 'religion', 'gender']] = df[['race', 'religion', 'gender']].replace(np.nan,'')

In [ ]:
df.isna().sum()

In [ ]:
train_df, val_df = train_test_split(df, test_size=.1, stratify=df['label'], random_state=42)

In [ ]:
x_train, y_train = train_df[['emoticon_1', 'emoticon_2', 'emoticon_3',
       'upvote', 'downvote', 'if_1', 'if_2', 'race', 'religion', 'gender',
       'disability', 'comment']], train_df['label']

In [ ]:
x_val, y_val = val_df[['emoticon_1', 'emoticon_2', 'emoticon_3',
       'upvote', 'downvote', 'if_1', 'if_2', 'race', 'religion', 'gender',
       'disability', 'comment']], val_df['label']

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.pipeline import Pipeline


In [ ]:
text_pipeline = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='')),
    ('reshape', FunctionTransformer(lambda x: x.ravel())),
    ('tfidf', TfidfVectorizer(preprocessor=clean_text, max_features=60000)),
    # ('to_numpy', FunctionTransformer(lambda x: x.toarray()))
    ])

In [ ]:
categorical_cols = ["race", "religion", "gender", "disability"]
categorical_pipeline = [("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))]


numerical_cols = ["emoticon_1", "emoticon_2", "emoticon_3","upvote", "downvote", "if_1", "if_2"]
numerical_pipeline = [("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())]

categorical_pipeline = Pipeline(categorical_pipeline)
numerical_pipeline = Pipeline(numerical_pipeline)


preprocessor = ColumnTransformer(
    transformers=[("cat", categorical_pipeline, categorical_cols),
        ("num", numerical_pipeline, numerical_cols),
        ("text", text_pipeline, ['comment'])],
    remainder="passthrough"
)

In [ ]:
x_train_matrix = preprocessor.fit_transform(x_train)

In [ ]:
x_val_matrix = preprocessor.transform(x_val)

from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=300, random_state=2306)

X_train_reduced = svd.fit_transform(x_train_matrix)
X_test_reduced = svd.transform(x_val_matrix)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier(n_estimators=2)
clf.fit(x_train_matrix, y_train)
y_pred = clf.predict(x_val_matrix)
confusion_matrix(y_val, y_pred)
accuracy_score(y_val, y_pred)

In [ ]:
from sklearn.neural_network import MLPClassifier
clf2 = MLPClassifier((300, 16, 32), verbose=True, max_iter=2,random_state=42, warm_start=True, activation='relu')

In [ ]:
clf2.fit(X_train_reduced, y_train)

In [ ]:
y_pred2 = clf2.predict(X_test_reduced)

confusion_matrix(y_val, y_pred2)

In [ ]:
accuracy_score(y_val, y_pred2)

In [ ]:
test_df = pd.read_csv('./kaggle/input/comment-category-prediction-challenge/test.csv')

In [ ]:
test_df

In [ ]:
test_df = test_df.drop(['created_date', 'post_id'], axis=1)

In [ ]:
x_test_m = preprocessor.transform(test_df)

In [ ]:
x_test = svd.transform(x_test_m)

In [ ]:
y_main = pd.read_csv('../submission_5.csv')['label']

In [ ]:
y_main_pred = clf2.predict(x_test)

In [ ]:
accuracy_score(y_main, y_main_pred)

In [ ]:
pd.DataFrame({'ID':list(range(1,len(test_df)+1)),'label':clf2.predict(x_test_m)}).to_csv('submission_pc.csv',index=False)

In [ ]:
pd.DataFrame({'ID':list(range(1,len(test_df)+1)),'label':clf.predict(x_test_m)}).to_csv('submission_pc_rf.csv',index=False)